# Fine-tune Vietnamese-biencoder with full train dataset, no split to 9:1 train & validation.
This notebook does:
- finds best top 200 results: best 100 from BM25, best 100 from Vietnamese Biencoder Embedder.
- uses raw bge-reranker-v2-m3 to get best 100 results from 200 results above.

The reason why I need to get 100 best results from 200 best results is I fine-tune bge-reranker-v2-m3 with hard negatives in ft-reranker.ipynb, and 200 results maybe have more results that have not been learned efficiently by fine-tuned bge-reranker-v2-m3 in ft-reranker.ipynb

In [1]:
!pip install py_vncorenlp sentence-transformers

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.4 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=05475ac75ef6250412c81ac99609ac7de34cb81d73c1dcfc94de977bca2ba6da
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [2]:
import json
import numpy as np
import pandas as pd
import py_vncorenlp

In [3]:
py_vncorenlp.download_model(save_dir='/kaggle/working')

--2026-03-02 03:55:11--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 1.09M 24s
    50K .......... .......... .......... .......... ..........  0% 3.88M 15s
   100K .......... .......... .......... .......... ..........  0% 1.89M 15s
   150K .......... .......... .......... .......... ..........  0% 5.87M 12s
   200K .......... .......... .......... .......... ..........  0% 7.91M 10s
   250K .......... .......... .......... .......... ..........  1% 2.73M 10s
   300K .......... .......... .......... .......... ..........  1% 8.67M 9s
   3

In [4]:
# Load the word and sentence segmentation component
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working')

text = "Ông Nguyễn Khắc Chúc  đang làm việc tại Đại học Quốc gia Hà Nội. Bà Lan, vợ ông Chúc, cũng làm việc tại đây."

output = rdrsegmenter.word_segment(text)

print(output)
# ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']

2026-03-02 03:55:25 INFO  WordSegmenter:24 - Loading Word Segmentation model
['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']


In [5]:
# segment laws in corpus
corpus_Id2Text = {}
corpus_Text2Id = {}


with open("/kaggle/input/datasets/duongquanganh/chunked-corpus/chunked_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

for raw in corpus:
    corpus_Id2Text[raw['chunk_id']] = raw['content_Article']
    corpus_Text2Id[raw['content_Article']] = raw['chunk_id']

In [6]:
with open("/kaggle/input/traindata-retrieval/train.json", "r", encoding = "utf-8") as f:
    train_data = json.load(f)

# segment question in train data
for q in train_data:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)

In [7]:
train = {'anchor': [], 'positive': []}
for q in train_data:
    for rel_law in q['relevant_laws']:
        pattern_rel_law = str(rel_law) + '_'
        i = 0
        while True:
            chunked_rel_law = pattern_rel_law + str(i)
            #print(chunked_rel_law)
            if chunked_rel_law not in corpus_Id2Text:
                break
            train['anchor'].append(q['question'])
            train['positive'].append(corpus_Id2Text[chunked_rel_law])
            i+=1

In [8]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers
from datasets import Dataset

train_dataset = Dataset.from_dict(train)

2026-03-02 03:55:45.250680: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772423745.437654      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772423745.489298      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772423745.947975      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772423745.948013      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772423745.948016      24 computation_placer.cc:177] computation placer alr

In [9]:
model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder',device='cuda')
loss = MultipleNegativesRankingLoss(model)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

In [10]:
import torch
torch.cuda.empty_cache()

In [11]:
args = SentenceTransformerTrainingArguments(
    output_dir="vnese-biencoder-encoder-MNRL",
    
    num_train_epochs=14,
    per_device_train_batch_size=100,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    fp16=True,
    bf16=False,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    logging_first_step=True,
    report_to="none",
    run_name="vnese-biencoder-encoder",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
1,1.358600
100,1.033200
200,0.578100
300,0.457400
400,0.418400
500,0.385600
600,0.375800
700,0.374800
800,0.355200
900,0.360200


TrainOutput(global_step=1260, training_loss=0.44287900953065784, metrics={'train_runtime': 3199.6129, 'train_samples_per_second': 39.087, 'train_steps_per_second': 0.394, 'total_flos': 0.0, 'train_loss': 0.44287900953065784, 'epoch': 14.0})

In [12]:
corpus_text = []

for raw in corpus:
    corpus_text.append(raw['content_Article'])

In [13]:
with open("/kaggle/input/privatetest/DRILL_PrivateTest/private_test.json","r",encoding = "utf-8") as f:
    file = json.load(f)
questions = []
idx_to_qid = {}
for idx,q in enumerate(file):
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)
    questions.append(q['question'])
    idx_to_qid[idx] = q['qid']

In [14]:
corpus_embeddings = model.encode(corpus_text)

In [15]:
corpus_embeddings = corpus_embeddings.astype(np.float32)

In [16]:
!pip install -Uq faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.9 MB/s eta 0:00:00


In [17]:
import faiss

dim = corpus_embeddings.shape[-1]

index = faiss.index_factory(dim, 'Flat', faiss.METRIC_INNER_PRODUCT)

In [18]:
index.train(corpus_embeddings)

In [19]:
print(index.is_trained)  

True


In [20]:
index.add(corpus_embeddings)

print(f"total number of vectors: {index.ntotal}")

total number of vectors: 97625


In [21]:
query_embeddings = model.encode(questions)

In [23]:
!pip install rank_bm25

In [24]:
from rank_bm25 import BM25Okapi

In [ ]:
top_k = 100
faiss_dists, faiss_ids = index.search(query_embeddings, k=top_k)

print(f"FAISS search completed: {len(faiss_ids)} queries, top {top_k} results each")

FAISS search completed: 627 queries, top 100 results each


In [ ]:
tokenized_corpus = [doc.split(" ") for doc in corpus_text]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index created with {len(tokenized_corpus)} documents")

BM25 index created with 97625 documents


In [ ]:
bm25_results = []

for question in questions:
    tokenized_query = question.split(" ")
    doc_scores = bm25.get_scores(tokenized_query)

    top_100_indices = np.argsort(doc_scores)[::-1][:top_k]
    bm25_results.append(top_100_indices)

bm25_results = np.array(bm25_results)
print(f"BM25 search completed: {len(bm25_results)} queries, top {top_k} results each")

BM25 search completed: 627 queries, top 100 results each


In [ ]:
combined_results = []

for idx in range(len(questions)):
    faiss_indices = set(faiss_ids[idx].tolist())
    bm25_indices = set(bm25_results[idx].tolist())

    combined_indices = faiss_indices.union(bm25_indices)

    combined_indices = list(combined_indices)
    
    combined_results.append({
        'qid': idx_to_qid[idx],
        'question': questions[idx],
        'candidate_indices': combined_indices,
        'num_candidates': len(combined_indices)
    })

print(f"Combined results for {len(combined_results)} questions")
print(f"Average candidates per question: {np.mean([r['num_candidates'] for r in combined_results]):.1f}")

Combined results for 627 questions
Average candidates per question: 173.6


In [ ]:
cross_encoder_input = []

for result in combined_results:
    qid = result['qid']
    question = result['question']
    
    candidate_texts = [corpus_text[idx] for idx in result['candidate_indices']]
    
    cross_encoder_input.append({
        'qid': qid,
        'question': question,
        'candidates': candidate_texts,
        'num_candidates': len(candidate_texts)
    })

print(f"Prepared {len(cross_encoder_input)} questions for cross-encoder re-ranking")
print(f"Example - Question 0 has {cross_encoder_input[0]['num_candidates']} candidate documents")

Prepared 627 questions for cross-encoder re-ranking
Example - Question 0 has 184 candidate documents


In [30]:
# Display a sample to verify the format
sample_idx = 0
print(f"Sample Question (qid: {cross_encoder_input[sample_idx]['qid']}):")
print(f"Question: {cross_encoder_input[sample_idx]['question'][:100]}...")
print(f"\nNumber of candidates: {cross_encoder_input[sample_idx]['num_candidates']}")
print(f"\nFirst candidate text:")
print(cross_encoder_input[sample_idx]['candidates'][1])

Sample Question (qid: 1497):
Question: Phạm_nhân không biết chữ có được tạo điều_kiện học văn_hoá nhằm xoá mù_chữ hay không ?...

Number of candidates: 184

First candidate text:
bên trái có tem bảo_mật , dưới tem có dòng chữ “ Số : … ” cỡ chữ 8 , màu đen ; bên phải có bốn dòng chữ , màu đen theo thứ_tự từ trên xuống : “ Hà_Nội , ngày .... tháng ... năm ... ” cỡ chữ 8 ; “ Hanoi , date .... month … .. year .... ” cỡ chữ 7 ; “ BỘ_TRƯỞNG BỘ CÔNG_AN ” cỡ chữ 8 ; “ MINISTER OF PUBLIC SECURITY ” cỡ chữ 6,5 . Có chữ_ký của Bộ_trưởng Bộ Công_an và đóng_dấu của Bộ Công_an ( có mẫu kèm theo ) . 3 . Thẩm_quyền cấp , thu_hồi Giấy chứng_nhận công_tác đặc_biệt . a ) Bộ_trưởng Bộ Công_an cấp Giấy chứng_nhận công_tác đặc_biệt và giao Tư_lệnh Bộ_tư_lệnh Cảnh_sát cơ_động , Giám_đốc Công_an tỉnh , thành_phố trực_thuộc trung_ương quản_lý , cấp , thu_hồi . b ) Giấy chứng_nhận công_tác đặc_biệt được cấp cho cán_bộ , chiến_sĩ Cảnh_sát cơ_động khi được giao thực_hiện phương_án tác_chiến chống khủng_bố và áp_t

In [ ]:
ce_input_data = cross_encoder_input

In [35]:
answer = []
for q in ce_input_data:
    answer.append({
        'qid': q['qid'],
        'relevant_laws': list(set(int(corpus_Text2Id[t].split("_")[0]) for t in q['candidates']))
    })

with open(f"answer_top200.json", "w", encoding="utf-8") as f:
        json.dump(answer, f, ensure_ascii=False)

# Use bge-reranker-v2-m3 raw for get best 100 results

In [ ]:
with open("/kaggle/input/datasets/duongquanganh/privatetest/DRILL_PrivateTest/private_test.json","r",encoding = "utf-8") as f:
    file = json.load(f)

qid_to_text = {}
text_to_qid = {}
for q in file:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)
    qid_to_text[q['qid']] = q['question']
    text_to_qid[q['question']] = q['qid']

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoder
model = CrossEncoder("BAAI/bge-reranker-v2-m3")

In [ ]:
# Re-rank candidates for each question
reranked_results = []

print(f"Re-ranking {len(ce_input_data)} questions...")

for i, item in enumerate(ce_input_data):
    qid = item['qid']
    question = item['question']
    candidates = item['candidates']
    
    pairs = [[question, candidate] for candidate in candidates]
    
    scores = model.predict(pairs)
    
    # Handle single candidate case (scores will be a float instead of list)
    if isinstance(scores, float):
        scores = [scores]
    
    # Create list of (candidate_text, score, original_index) tuples
    candidate_score_pairs = [(candidates[j], scores[j], j) for j in range(len(candidates))]
    
    # Sort by score in descending order (highest score first)
    candidate_score_pairs.sort(key=lambda x: x[1], reverse=True)
    
    # Store reranked results
    reranked_results.append({
        'qid': qid,
        'question': question,
        'reranked_candidates': [
            {
                'chunk_id': corpus_Text2Id[text],
                'score': score,
                'original_rank': orig_idx
            }
            for text, score, orig_idx in candidate_score_pairs
        ],
        'num_candidates': len(candidates)
    })
    
    # Print progress every 10 questions
    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(ce_input_data)} questions")

print(f"\nRe-ranking complete! Processed {len(reranked_results)} questions")

In [ ]:
top_100_results = []

for result in reranked_results:
    top_100_results.append({
        'qid': result['qid'],
        'relevant_laws': list(set(int(t['chunk_id'].split("_")[0]) for t in result['reranked_candidates'][:100]))
    })

with open("reranked_top100_results_submit.json", "w", encoding="utf-8") as f:
    json.dump(top_100_results, f, ensure_ascii=False, indent=2)

print(f"Saved top 100 reranked results for {len(top_100_results)} questions to 'reranked_top100_results_submit.json'")

In [ ]:
top_100_results_input_rerank = []

for result in reranked_results:
    top_100_results_input_rerank.append({
        'qid': result['qid'],
        'relevant_laws': list(t['chunk_id'] for t in result['reranked_candidates'][:100])
    })

with open("reranked_top100_results_input_rerank.json", "w", encoding="utf-8") as f:
    json.dump(top_100_results_input_rerank, f, ensure_ascii=False, indent=2)

print(f"Saved top 100 reranked results for {len(top_100_results_input_rerank)} questions to 'reranked_top100_results_input_rerank.json'")